[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/03_setup/03_setup_check.ipynb)


# 03 — 환경 구성 — 설치·검증·러닝 예제

> 본문 [`03_setup.md`](03_setup.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** ANARCI numbering+germline **0.15초** · Sapiens 첫 실행(모델 가중치 다운로드 포함) **6.3초**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `03_setup/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `03_setup/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "03_setup"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — 도구 설치 (본문 3.1~3.2)

설치 채널이 도구마다 달라요. **BioPhi 는 bioconda 전용**(PyPI 에 없음), **Humatch 는 GitHub source**, Sapiens·AnthroAb·AbNatiV 는 PyPI 예요.
채널 지도 전체는 4) 의 표(`data/verified_versions.csv` 의 `install` 컬럼)에서 한 번에 봅니다.

로컬 conda 라면 본문 3.1 의 한 줄로 바닥을 깔 수 있어요.

```bash
conda create -n abhuman -c conda-forge -c bioconda python=3.10 anarci hmmer -y
conda activate abhuman
ANARCI --help
```

이 노트북이 이 챕터에서 실제로 하는 일은 넷이에요 — **① `anarci`·`abnumber`·`sapiens` 설치 → ② import 진단 → ③ 예제 서열 numbering → ④ 검증 버전 대조**.
`hmmscan`(HMMER) 은 위 부트스트랩 셀의 apt 단계에서 이미 깔렸어요. 설치가 실패해도 랩은 멈추지 않아요 — 2) 진단표가 무엇이 빠졌는지 짚어 줍니다.

In [ ]:
t0 = time.time()
try:
    _ensure("anarci abnumber sapiens")          # 이미 있으면 건너뜁니다
    print(f"설치 셀 소요 {time.time()-t0:.1f}초")
except Exception as e:
    print("설치 실패:", type(e).__name__, str(e)[:200])
    print("→ 여기서 멈추지 마세요. 2) 진단표에서 FAIL 인 줄만 골라 그 줄의 설치 명령을 다시 돌리면 돼요.")

print("hmmscan :", shutil.which("hmmscan") or "PATH 에 없음 → ANARCI numbering 이 FileNotFoundError 로 죽어요")

## 2) 내 결과 확인 — 환경 진단 (본문 3.3·3.4)

본문 3.4 의 검증 네 줄(`ANARCI --help` · `import sapiens` · `import anthroab` · `import anarci`)을 파이썬에서 한 번에 확인해요.
마지막 줄이 특히 중요해요. CLI `ANARCI` 가 잘 돌아도 **파이썬 모듈 `anarci` 가 import 되지 않는** 상태가 있거든요.
본문 3.3 대로 `anarci` 는 **ANARCI 자신 · Humatch 의 정렬 · AbNatiV 의 `-align`** 세 곳이 공통으로 import 해요 — env 를 나눌 거면 **각 env 에 넣어야** 합니다.

In [ ]:
import pandas as pd

def probe(mod):
    try:
        m = importlib.import_module(mod)
        return "OK", getattr(m, "__version__", "?")
    except Exception as e:
        return "FAIL", f"{type(e).__name__}: {e}"[:60]

CHECKS = [("anarci",   "필수", "ANARCI 자신 · Humatch 정렬 · AbNatiV -align 이 공통 import (본문 3.3)"),
          ("abnumber", "필수", "CDR/region 추출 (Ch.04)"),
          ("sapiens",  "필수", "humanization 엔진 (Ch.05)"),
          ("anthroab", "나중", "본문 3.4 의 검증 4줄 중 하나 — 설치는 Ch.06 에서 해도 돼요")]

rows = []
for mod, when, why in CHECKS:
    st, ver = probe(mod)
    rows.append({"항목": f"import {mod}", "언제": when, "상태": st, "버전/메시지": ver, "쓰는 곳": why})
rows.append({"항목": "hmmscan (HMMER)", "언제": "필수",
             "상태": "OK" if shutil.which("hmmscan") else "FAIL",
             "버전/메시지": shutil.which("hmmscan") or "PATH 에 없음",
             "쓰는 곳": "ANARCI 가 numbering 을 이 서브프로세스로 돌려요"})
rows.append({"항목": "ANARCI CLI", "언제": "권장",
             "상태": "OK" if shutil.which("ANARCI") else "FAIL",
             "버전/메시지": shutil.which("ANARCI") or "없음",
             "쓰는 곳": "Ch.04 의 --assign_germline (파이썬 API 로도 진행 가능)"})

env = pd.DataFrame(rows)
display(env)

bad = [r["항목"] for r in rows if r["언제"] == "필수" and r["상태"] == "FAIL"]
if bad:
    print("필수 항목 실패 —", ", ".join(bad))
    print("→ import 실패는 1) 셀 재실행, hmmscan 실패는 Colab 이면 apt-get install -y hmmer 로 해결돼요.")
else:
    print("필수 4줄(anarci · abnumber · sapiens · hmmscan) 전부 OK — 3) 으로 넘어가도 좋아요.")
print("'나중' 인 항목은 지금 FAIL 이어도 정상이에요 — 해당 챕터에서 설치합니다.")

## 3) 직접 실행 — 러닝 예제 서열로 첫 numbering

가이드 전체를 관통하는 예제 서열(`data/parental.fasta`)이에요. Ch.04~10 의 모든 수치가 이 두 체인에서 나옵니다.
여기서는 **numbering 경로가 실제로 도는지**만 확인해요 — CDR 표·germline 할당은 Ch.04 에서 제대로 합니다.

In [ ]:
try:
    from abnumber import Chain

    t0 = time.time()
    ch_h = Chain(VH, scheme="imgt")
    ch_l = Chain(VL, scheme="imgt")
    el = time.time() - t0

    print(f"VH  : {len(VH)} aa | chain_type={ch_h.chain_type} | CDR3={ch_h.cdr3_seq}")
    print(f"VL  : {len(VL)} aa | chain_type={ch_l.chain_type} | CDR3={ch_l.cdr3_seq}")
    print(f"abnumber numbering(2 체인) 소요 {el:.2f}초")

    ok = (ch_h.chain_type == "H") and (ch_l.chain_type in ("K", "L")) and bool(ch_h.cdr3_seq) and bool(ch_l.cdr3_seq)
    print("판정 —", "chain_type H/L 이 제대로 붙고 CDR3 두 개가 나왔어요. numbering 경로(hmmscan 포함) 정상이에요."
          if ok else "chain_type 이나 CDR3 가 비었어요. 2) 진단표의 FAIL 줄부터 해결하세요.")
except Exception as e:
    print("numbering 실패:", type(e).__name__, str(e)[:160])
    print("→ 2) 진단표에서 FAIL 인 항목을 보고 1) 설치 셀을 다시 실행하세요.")
    print("   hmmscan 이 없으면 numbering 이, abnumber 가 없으면 CDR 추출이 여기서 막혀요.")

## 4) 레퍼런스 대조 — 이 가이드를 검증한 도구 9종 (본문 3.2)

`data/verified_versions.csv` 는 이 가이드의 모든 수치를 뽑을 때 **실제로 쓴 버전과 설치 채널**이에요.
버전이 달라도 대개 문제없지만, 뒤 챕터의 결과가 어긋나면 여기부터 비교하세요.

In [ ]:
ver = pd.read_csv(find_one("verified_versions.csv"))

IMPORT_OF = {"ANARCI": "anarci", "Humatch": "humatch"}       # 표기 ≠ import 이름
WHERE     = {"ANARCI": "Ch.04·06·07", "abnumber": "Ch.04", "sapiens": "Ch.05",
             "Humatch": "Ch.06", "anthroab": "Ch.06", "abnativ": "Ch.07",
             "igfold": "Ch.08", "transformers": "Ch.07", "torch": "Ch.07·08"}
LIGHT     = {"anarci", "abnumber", "sapiens", "anthroab"}    # 버전 확인을 위해 import 해도 싼 것들

from importlib.metadata import version as _dist_version

def installed_version(mod):
    for nm in (mod, mod.lower(), mod.upper()):
        try:
            return _dist_version(nm)
        except Exception:
            pass
    if not _have(mod):
        return None
    if mod in LIGHT:
        try:
            return getattr(importlib.import_module(mod), "__version__", "설치됨")
        except Exception:
            return "설치됨(버전 미상)"
    return "설치됨"

rows = []
for _, r in ver.iterrows():
    tool = str(r["tool"])
    mod  = IMPORT_OF.get(tool, tool.lower())
    got  = installed_version(mod)
    rows.append({"도구": tool, "검증 버전": r["version"], "내 버전": got or "미설치",
                 "상태": "미설치" if got is None else ("일치" if str(got) == str(r["version"]) else "다름"),
                 "설치 채널": r["install"], "쓰는 곳": WHERE.get(tool, "-")})
vt = pd.DataFrame(rows)
display(vt)

need = vt[vt["도구"].isin(["ANARCI", "abnumber", "sapiens"])]
miss = list(need.loc[need["상태"] == "미설치", "도구"])
print("이 챕터 필수 3종 —", "전부 설치됨" if not miss else "빠짐 " + ", ".join(miss))
print(f"나머지 {len(vt) - len(need)}종은 쓰는 챕터에서 설치돼요 — 지금 '미설치' 여도 정상이에요.")

trow = vt[vt["도구"] == "torch"]
if len(trow):
    want, got = str(trow["검증 버전"].iloc[0]), str(trow["내 버전"].iloc[0])
    note = str(ver.loc[ver["tool"] == "torch", "note"].iloc[0])
    if got == "미설치":
        print(f"torch — 아직 없어요. Ch.07·08 에서 검증 버전 {want} 로 깔면 돼요.")
    elif got == want:
        print(f"torch {got} — 검증 버전과 같아요.")
    else:
        print(f"torch {got} ≠ 검증 {want} — {note}")

## 이 랩에서 확인한 것

1. `hmmscan`(HMMER) 이 PATH 에 없으면 ANARCI numbering 이 **FileNotFoundError** 로 죽어요 — Colab 은 `apt-get install -y hmmer` 가 정답.
2. **`import anarci` 는 세 곳(ANARCI·Humatch·AbNatiV `-align`)이 공통으로 쓰는 관문**이에요. env 를 나누면 각 env 에 넣으세요.
3. 설치 채널은 도구마다 달라요 — BioPhi 는 **bioconda 전용**, Humatch 는 **GitHub source**, Sapiens·AnthroAb·AbNatiV 는 **PyPI**. 4) 의 `설치 채널` 컬럼이 9종 전체 지도예요.
4. 러닝 예제 CDR3 — VH `ARRGRYGLYAMDY` · VL `QSYDSSLRVV`. 이 값이 나왔다면 Ch.04 로 넘어가도 좋아요.

다음 → **[Ch.04 — 입력 QC](../04_sequence_qc/04_numbering_lab.ipynb)**